In [ ]:
from pytao import Tao
from pprint import pprint
import json
import pandas as pd

In [ ]:
tao = Tao('-init $LCLS_LATTICE/bmad/models/sc_sxr/tao.init -noplot')

In [ ]:
IXLIST = tao.lat_list('*', 'ele.ix_ele', flags='-array_out -index_order -no_slaves -track_only')
IXLIST

In [ ]:
def ele_info(ele):
    dat = tao.ele_head(ele)
    dat.update(tao.ele_gen_attribs(ele))
    return dat
df = pd.DataFrame(map(ele_info, IXLIST), index=IXLIST)
df

In [ ]:
SLIST = df.s.fillna(0)
LLIST = df.L.fillna(0)
NAMES = df.name


ix_of = {}
for ix, name in enumerate(NAMES):
    if name in ix_of:
        pass
#        print("err", name)
    else: 
        ix_of[name] = ix

for s, l, n in zip(SLIST[0:20], LLIST[0:20], NAMES[0:20]):
    print (n, l, s)

In [ ]:
# Search for split eles

def find_split_eles(slist, llist, names):
    """
    Searches for split elements
    
    """

    split_eles = {}
    basename = 'xxx'
    split_eles[basename] = {'stubs':[], 'splits':[], 'offsets':[]}
    stubs  = split_eles[basename]['stubs']
    splits = split_eles[basename]['splits']
    offsets = split_eles[basename]['offsets']
    s0 = 0
    for s, l, n in zip(SLIST, LLIST, NAMES):
        
        if l == 0:
            splits.append(n)
            offsets.append(s - s0)
        # Thick ele
        elif n.startswith(basename) and len(n)==len(basename)+1:
            # this is a true split ele     
            stubs.append(n)
            split_eles[basename]['L'] += l
        else:
            # New thick ele
            
            # Removing unwanted eles
            if basename == 'K21_3B':
                # Special case. Leave these
                print(basename, stubs, splits)
                pass
            elif len(stubs) == 1 or len(splits) == 0:
                split_eles.pop(basename)
            # Or if this is obviously a superimpose element
            elif basename.endswith('#'):
                split_eles.pop(basename)
                
            # Try a basename which excludes the final character
            basename = n[:-1]
            split_eles[basename] = {'stubs':[], 'splits':[], 'offsets':[]}
            split_eles[basename]['L'] = l
            s0 = s -l 
            stubs  = split_eles[basename]['stubs']
            splits = split_eles[basename]['splits']
            offsets = split_eles[basename]['offsets']
            stubs.append(n) # add this 
    
        #print (n, l, s, ix)
        
    split_eles.pop('xxx')        
    
    return split_eles

SPLIT_ELES = find_split_eles(SLIST, LLIST, NAMES)
    
pprint(SPLIT_ELES )

In [ ]:
pprint(SC_LINAC_REPLACEMENTS)

In [ ]:
BENDS_TO_DESPLIT = [
    'bxh1', 'bxh2', 'bxh3', 'bxh4',
    'bx01', 'bx02', 
    'bx11', 'bx12', 'bx13', 'bx14',
    'bx21', 'bx22', 'bx23', 'bx24']
def desplit_bend_line(name):
    return f'{name}_full: line = ({name})'

BEND_REPLACEMENTS = {}
for name in BENDS_TO_DESPLIT:
    BEND_REPLACEMENTS[name+'_full'] = desplit_bend_line(name)
BEND_REPLACEMENTS 


# Desplit devel

In [1]:
# Useful for debugging
%load_ext autoreload
%autoreload 2

In [2]:
# Patch in the slac2bmad package
import sys
sys.path.append('python')
from slac2bmad.desplit import desplit_eles, desplit_ele

In [3]:
#LINE =  "CAVL354_full : LINE=(DCAVMAP,CAVL354,DCAVMAP)"
LINE =  "CAVL355_full : LINE=(DCAVMAP,CAVL355a,CSP35,CAVL355b,DCAVMAP)"

In [20]:
print(desplit_ele(LINE))

Special padded desplit, inside names end with a,b: CAVL355_full : LINE=(DCAVMAP,CAVL355a,CSP35,CAVL355b,DCAVMAP)
! Old split line: CAVL355_full : LINE=(DCAVMAP,CAVL355a,CSP35,CAVL355b,DCAVMAP)   
cavl355_full: line = (dcavmap, cavl355, dcavmap)
    csp35[superimpose] = T
    csp35[ref] = cavl355
    csp35[ele_origin] = beginning
    csp35[ref_origin] = beginning
    csp35[offset] = cavl355a[L] 
    


In [ ]:
def desplit_cavity(line):
    ele, line = line.strip().split(':')
    ele = ele.strip().split('_full')[0].lower()
    eles = [e.strip().lower() for e in (line.split('(')[1].split(')')[0]).split(',')]
    
    return ele, eles
    return process_padded_desplit(ele, eles)
print(desplit_cavity(LINE))

In [21]:
def process_padded_desplit(name, eles, note=""):
    """
    
    Example:
        process_padded_desplit('cavl355', ['dcavmap', 'cavl355a', 'csp35', 'cavl355b', 'dcavmap'])
    returns:
    !
    cavl355_full: line = (dcavmap, cavl355, dcavmap)
        csp35[superimpose] = T
        csp35[ref] = cavl355
        csp35[ele_origin] = beginning
        csp35[ref_origin] = beginning
        csp35[offset] = cavl355a[L]
    
    """
    
    pad = eles[0]
    assert eles[-1] == pad
    assert eles[1].startswith(name), eles[1]
    sele = eles[2]
    assert eles[3].startswith(name), eles[3]
    
    line = f"""! {note}   
{name}_full: line = ({pad}, {name}, {pad})
    {sele}[superimpose] = T
    {sele}[ref] = {name}
    {sele}[ele_origin] = beginning
    {sele}[ref_origin] = beginning
    {sele}[offset] = {eles[1]}[L] 
    """

    return line


print(process_padded_desplit('cavl355', ['dcavmap', 'cavl355a', 'csp35', 'cavl355b', 'dcavmap'], "some note"))
    

! some note   
cavl355_full: line = (dcavmap, cavl355, dcavmap)
    csp35[superimpose] = T
    csp35[ref] = cavl355
    csp35[ele_origin] = beginning
    csp35[ref_origin] = beginning
    csp35[offset] = cavl355a[L] 
    


In [ ]:
 6.5922156E-01 3.7852156E-01

In [ ]:
6.5922156E-0